In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))
from graph import Graph
import json
from itertools import islice  

In [49]:
# read uniprot synonym name dictionary
# with open("uniprot_synonym_mapping.json", "r", encoding="utf-8") as f: 
with open("../Databases/labelGraph_new_evid.json", "r", encoding="utf-8") as f:
    synonym_mapping_dict = json.load(f)

In [50]:
# read omic network
with open('../Databases/universalGraph_new_evid.json', 'r') as file:
    data = json.load(file)

G = Graph()
for vertex in data['vertices']:
    G.add_vertex(vertex, label=data['vertices'][vertex]['label'],
                 vert_info=data['vertices'][vertex]['vert_info'],
                 omic_type=data['vertices'][vertex]['omic_type'])
for edge in data['edges']:
    G.add_edge(start_id=edge['start_vertex'], end_id=edge['end_vertex'], int_info=edge['int_info'])

In [51]:
print("Number of nodes:", len(G.get_vertices()))
print("Number of edges:", len(G.get_edges()))

Number of nodes: 21048
Number of edges: 126535


In [6]:
from collections import defaultdict

unique_combinations = set()

# Group interaction types by start vertex
groups = defaultdict(set)  # start_vertex_id -> set of interaction types
for node in list(G.get_vertices()):
    for edge in G.get_vertex(node).get_inbound_edges():
        start_id = edge.get_end_vertex().get_id()
        interaction_type = edge.get_int_info()["interaction"]["0"]
        groups[start_id].add(interaction_type)
    
# Convert each group's set to an immutable frozenset so it can be added to a set
for interaction_set in groups.values():
    # Remove duplicates like multiple "Activation"
    cleaned = set(interaction_set)
    # Example filtering: remove 'part_of' if only 'Activation' and 'part_of' exist?
    unique_combinations.add(frozenset(cleaned))

print(unique_combinations)

{frozenset({'part_of'}), frozenset({'part_of', 'Activation'}), frozenset({'part_of', 'Repression', 'Activation'}), frozenset({'Repression', 'translated_to'}), frozenset({'Activation', 'Repression', 'translated_to'}), frozenset({'catalyzes'}), frozenset({'part_of', 'Repression'}), frozenset({'translated_to'}), frozenset({'translated_to', 'Activation'}), frozenset({'transcribed_to'})}


In [6]:
for edge in G.get_vertex('COMPLEX:P62987_Q96B02').get_outbound_edges() :
    print(edge.get_start_vertex().get_id() + " -> " + edge.get_int_info()["interaction"]["0"] + 
          " -> ", edge.get_end_vertex().get_id())

COMPLEX:P62987_Q96B02 -> Activation ->  Q9Y4K3_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q9BV68_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q8IUD6_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q9HAU4_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q8NHY2_protein
COMPLEX:P62987_Q96B02 -> Activation ->  O14686_protein
COMPLEX:P62987_Q96B02 -> Activation ->  P29590_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q7Z569_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q96EP1_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q9Y577_protein
COMPLEX:P62987_Q96B02 -> Repression ->  Q06587_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q96GF1_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q8WV44_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q13129_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q9HCE7_protein
COMPLEX:P62987_Q96B02 -> Activation ->  P38398_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q9UM63_protein
COMPLEX:P62987_Q96B02 -> Activation ->  Q03112_protein
COMPLEX:P6

In [19]:
val_act = 0
val_translated = 0
val_part_of = []
val_act -= 0.00745844688377062
if val_act :
    print(val_act) 
else : 
    print("None")

-0.00745844688377062


In [21]:
values_to_min = [v for v in (val_act if val_act else None,
                             val_translated if val_translated else None)
                 if v is not None] + val_part_of
print(values_to_min)
if not values_to_min:
    print("Not")

[-0.00745844688377062]


In [22]:
values_to_min = []
if not values_to_min:
    print("Not")

Not


In [5]:
from collections import defaultdict

omic_type_counts = defaultdict(int)

for v in G.get_vertices().values():
    omic_type = v.get_omic_type()
    omic_type_counts[omic_type] += 1

# Print all omic types and their counts
for omic_type, count in omic_type_counts.items():
    print(f"{omic_type}: {count}")

R: 5938
gene: 5666
transcript: 5666
protein: 5666
miRNA: 2656
protein_complex: 523


In [52]:
vertices_id = set(G.get_vertices().keys())

In [53]:
def matched_genes(gene_data, vertices_id, synonym_mapping_dict):
    columns = gene_data.columns[1:]  # Skip first column (e.g., sample ID)
    matched_columns = {}

    for col in columns:
        uniprots = synonym_mapping_dict.get(col)
        if not uniprots:
            continue

        for uniprot_id in uniprots:
            transcript_id = f"{uniprot_id}_transcript"
            transcript_x_id = f"{uniprot_id}_transcript_x"

            if transcript_id in vertices_id:
                matched_columns.setdefault(transcript_id, []).append(col)
                break

            elif transcript_x_id in vertices_id:
                matched_columns.setdefault(transcript_x_id, []).append(col)
                break

    return matched_columns

In [54]:
def matched_genes_label(gene_data, vertices_id, synonym_mapping_dict):
    columns = gene_data.columns[1:]  # Skip first column (e.g., sample ID)
    matched_columns = {}

    for col in columns:
        uniprot_id = synonym_mapping_dict.get(col)
        if not uniprot_id:
            continue

        transcript_id = f"{uniprot_id}_transcript"
        transcript_x_id = f"{uniprot_id}_transcript_x"

        if transcript_id in vertices_id:
            matched_columns.setdefault(transcript_id, []).append(col)

        elif transcript_x_id in vertices_id:
            matched_columns.setdefault(transcript_x_id, []).append(col)

    return matched_columns

In [55]:
cancer_names = ['breast','ccRCC3', 'ccRCC4', 'coad', 'pdac', 'prostat', 'AD']
directory = '../Databases'

gene_data = {}
mapped_genes = {}
mapping_counts = {}

def count_multi_mappings(mapping):
    return len([v for v in mapping.values() if len(v) > 1])
    
for name in cancer_names:
    file_path = os.path.join(directory, f'geneData_{name}.csv')
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    # Load data
    df = pd.read_csv(file_path)
    gene_data[name] = df

    # Map genes
    mapped = matched_genes_label(df, vertices_id, synonym_mapping_dict)
    mapped_genes[name] = mapped

    # Count total and ambiguous mappings
    total = len(mapped)
    ambiguous = count_multi_mappings(mapped)
    mapping_counts[name] = [len(df.columns)-1, total, ambiguous]
    #break

In [56]:
for k,v in mapped_genes.items() :
    for z, t in v.items() :
        if len(t) > 1 :
            print("key : ", z, " value : ", t)
    break
#mapping_counts

In [57]:
mapping_counts

{'breast': [16382, 3245, 0],
 'ccRCC3': [22928, 4040, 0],
 'ccRCC4': [23001, 4041, 0],
 'coad': [21634, 4033, 0],
 'pdac': [20032, 3996, 0],
 'prostat': [27827, 4019, 0],
 'AD': [55889, 3107, 0]}

In [18]:
print("5 Samples from matched genes:")
for gene, vertex in islice(mapped_genes['breast'].items(), 5):
    print(f"{gene} → {vertex}")
print("*****************************")
print("Number of mached genes : ", len(mapped_genes['breast']))

5 Samples from matched genes:
*****************************
Number of mached genes :  0


In [43]:
import random

def random_walk_to_reaction_with_details(graph, start_id, max_steps=50):
    if start_id not in graph.get_vertices():
        print(f"{start_id} grafikte yok.")
        return

    current = graph.get_vertices()[start_id]
    path_parts = [f"{current.get_id()}({current.get_omic_type()})"]

    for _ in range(max_steps):
        outbound = list(current.get_outbound_edges())
        if not outbound:
            print("Çıkış kenarı kalmadı.")
            break

        edge = random.choice(outbound)

        # type bilgisi
        type_info = edge.get_int_info().get("type", "")

        # interaction bilgisi
        interaction_info = ""
        if "interaction" in edge.get_int_info():
            interaction_field = edge.get_int_info()["interaction"]
            if isinstance(interaction_field, dict):
                interaction_info = ", ".join(interaction_field.values())
            elif isinstance(interaction_field, str):
                interaction_info = interaction_field

        target = edge.get_end_vertex()

        path_parts.append(
            f"-[{interaction_info}]-> {target.get_id()}({target.get_omic_type()})"
        )

        if target.get_omic_type().upper() == "R":
            print("Reaction bulundu!")
            print(" ".join(path_parts))
            return

        current = target

    print("Reaction bulunamadı.")
    #print(" ".join(path_parts))


# Kullanım:
random_walk_to_reaction_with_details(G, "COMPLEX:P05412_P15336")

Reaction bulundu!
COMPLEX:P05412_P15336(protein_complex) -[Activation]-> P01574_protein(protein) -[Activation]-> P17181_protein(protein) -[Activation]-> COMPLEX:P27986_P42336(protein_complex) -[Activation]-> P42338_protein(protein) -[catalyzes]-> PIK3(R)


In [ ]:
matches = find_complex_protein_reaction(filtered_graph)
for m in matches:
    print(f"Complex {m['complex']} activates Protein {m['protein']} "
          f"which catalyzes Reaction {m['reaction']}")

In [16]:
a = None
b = 5
c = 42

In [15]:
min(a,b,c)

2

In [25]:
if [5] :
    print("Ben 5")

Ben 5


In [43]:
val_act = 10
val_translated = 0
val_part_of = [8]
values_to_min = [v for v in (val_act if val_act else None,
                             val_translated if val_translated else None)
                 if v is not None] + val_part_of

In [44]:
values_to_min

[10, 8]